# Optimisation de la valeur de stock — Horizon restant de l'année fiscale

## 1. Contexte et problématique

Nous sommes en **mai** : il reste **5 mois** avant la clôture de l'année fiscale (mai à
septembre). Un état des lieux du stock, portant sur **15 références**, fait apparaître une
valeur totale actuelle d'environ **5,83 M€**, largement au-dessus de la cible de fin d'année.

L'objectif est de déterminer, pour chaque référence, **quand et combien commander** (ou
s'abstenir de commander) sur les 5 mois restants, de façon à :
- satisfaire la demande client à chaque période,
- respecter un stock de sécurité minimal (à partir du 2ᵉ mois),
- **réduire la valeur totale du stock d'environ 2 M€** d'ici fin septembre,
- tenir compte de deux contraintes opérationnelles :
  - **la fermeture des fournisseurs en août** (aucune commande ne peut être passée ce mois-là,
    même si elle peut être reçue si elle a été passée avant),
  - **une capacité de stockage limitée** (en m³).

Contrairement à un scénario où la seule décision rationnelle serait de ne rien commander, ce
jeu de données inclut des références dont le stock actuel est **insuffisant** pour couvrir la
demande à venir : le modèle doit donc arbitrer entre désengagement (laisser filer le stock
excédentaire) et réapprovisionnement ciblé (commander là où c'est nécessaire), sous contrainte
de MOQ et de fermeture fournisseur.

## 2. Formulation mathématique

### Ensembles
| Symbole | Signification |
|---|---|
| $I$ | ensemble des références produit ($\lvert I \rvert = 15$) |
| $T = \{1,\dots,5\}$ | périodes (1=Mai, 2=Juin, 3=Juillet, 4=Août, 5=Septembre) |
| $A \subset T$ | périodes de fermeture fournisseur ($A = \{4\}$, Août) |

### Paramètres
| Symbole | Signification |
|---|---|
| $p_i$ | prix unitaire de la référence $i$ |
| $S^0_i$ | stock initial (mai) |
| $d_{i,t}$ | demande de la référence $i$ en période $t$ |
| $LT_i$ | délai de livraison (en mois) |
| $MOQ_i$ | quantité minimale de commande |
| $v_i$ | volume unitaire de stockage (m³) |
| $SS_i$ | stock de sécurité |
| $c^s_i$ | coût de possession unitaire par mois |
| $c^o_i$ | coût fixe de passation d'une commande |
| $TARGET$ | cible de valeur de stock en fin d'horizon |
| $V_{max}$ | capacité totale de stockage (m³) |
| $M$ | grand-M (borne supérieure large sur les commandes) |

### Variables de décision
| Symbole | Type | Signification |
|---|---|---|
| $S_{i,t} \ge 0$ | continue | stock de $i$ en fin de période $t$ |
| $O_{i,t} \ge 0$ | continue | quantité commandée de $i$ en période $t$ |
| $z_{i,t} \in \{0,1\}$ | binaire | 1 si une commande de $i$ est passée en $t$ |

### Fonction objectif

Minimiser le coût total de possession et de passation de commandes :
$$
\min \quad Z = \sum_{i \in I}\sum_{t \in T} c^s_i \, S_{i,t} \;+\; \sum_{i \in I}\sum_{t \in T} c^o_i \, z_{i,t}
$$

### Contraintes

**C1 — Bilan de stock** :
$$
S_{i,t} = S_{i,t-1} + O_{i,\,t-LT_i}\cdot\mathbb{1}[t-LT_i \ge 1] - d_{i,t} \qquad \forall i \in I,\ t \in T
$$

**C2 — Quantité minimale de commande (MOQ)**, liée à la variable binaire $z_{i,t}$ :
$$
MOQ_i \cdot z_{i,t} \;\le\; O_{i,t} \;\le\; M \cdot z_{i,t} \qquad \forall i \in I,\ t \in T
$$

**C3 — Stock de sécurité**, à partir de $t=2$ (le stock initial est une donnée subie, pas une
décision — l'exiger dès $t=1$ rendrait le modèle infaisable par construction) :
$$
S_{i,t} \ge SS_i \qquad \forall i \in I,\ t \in T,\ t \ge 2
$$

**C4 — Fermeture fournisseur en août** :
$$
O_{i,t} = 0 \qquad \forall i \in I,\ t \in A
$$

**C5 — Capacité de stockage** :
$$
\sum_{i \in I} v_i \, S_{i,t} \;\le\; V_{max} \qquad \forall t \in T
$$

**C6 — Cible de valeur de stock en fin d'horizon** ($T_{max}=5$, Septembre) :
$$
\sum_{i \in I} p_i \, S_{i,T_{max}} \;\le\; TARGET
$$


In [ ]:
from pyomo.environ import *
from IPython.display import display
import pandas as pd
import numpy as np
import math


---
## 3. Implémentation — mêmes 5 étapes que la démarche AMPL/GUSEK

### 1. LES PARAMÈTRES


In [ ]:
model = AbstractModel()

model.T   = Param(within=PositiveIntegers)
model.PER = RangeSet(1, model.T)
model.AoutSet = Set(within=model.PER)   # C4

model.I = Set()
model.M = Param()

model.prix          = Param(model.I)   # p_i
model.stock_init    = Param(model.I)   # S0_i
model.LT            = Param(model.I)
model.MOQ           = Param(model.I)
model.volume        = Param(model.I)   # v_i
model.SS            = Param(model.I)
model.cout_stockage = Param(model.I)   # c^s_i
model.cout_commande = Param(model.I)   # c^o_i

model.demande = Param(model.I, model.PER)

model.TARGET = Param()
model.V_max  = Param()


### 2. LES VARIABLES

In [ ]:
model.S = Var(model.I, model.PER, domain=NonNegativeReals)   # S_{i,t}
model.O = Var(model.I, model.PER, domain=NonNegativeReals)   # O_{i,t}
model.z = Var(model.I, model.PER, domain=Binary)              # z_{i,t}


### 3. LA FONCTION OBJECTIF (et les contraintes C1 → C6)

In [ ]:
def cost_rule(m):
    return (sum(m.cout_stockage[i]*m.S[i,t] for i in m.I for t in m.PER)
          + sum(m.cout_commande[i]*m.z[i,t]  for i in m.I for t in m.PER))
model.cost = Objective(rule=cost_rule, sense=minimize)


In [ ]:
# C1 — Bilan de stock
def c1_rule(m, i, t):
    S_prev = m.stock_init[i] if t == 1 else m.S[i, t-1]
    t_cmd  = t - m.LT[i]
    reception = m.O[i, t_cmd] if t_cmd >= 1 else 0
    return m.S[i,t] == S_prev + reception - m.demande[i,t]
model.C1_Bilan = Constraint(model.I, model.PER, rule=c1_rule)

# C2 — MOQ
def c2a_rule(m, i, t):
    return m.O[i,t] >= m.MOQ[i] * m.z[i,t]
model.C2a_MOQ_min = Constraint(model.I, model.PER, rule=c2a_rule)

def c2b_rule(m, i, t):
    return m.O[i,t] <= m.M * m.z[i,t]
model.C2b_MOQ_max = Constraint(model.I, model.PER, rule=c2b_rule)

# C3 — Stock de sécurité, à partir de t=2
def c3_rule(m, i, t):
    if t == 1:
        return Constraint.Skip
    return m.S[i,t] >= m.SS[i]
model.C3_StockSecurite = Constraint(model.I, model.PER, rule=c3_rule)

# C4 — Fermeture fournisseur en août
def c4_rule(m, i, t):
    return m.O[i,t] == 0
model.C4_FermetureFournisseur = Constraint(model.I, model.AoutSet, rule=c4_rule)

# C5 — Capacité de stockage
def c5_rule(m, t):
    return sum(m.volume[i]*m.S[i,t] for i in m.I) <= m.V_max
model.C5_CapaciteStockage = Constraint(model.PER, rule=c5_rule)

# C6 — Cible de valeur de stock en fin d'horizon
def c6_rule(m):
    return sum(m.prix[i]*m.S[i, m.T] for i in m.I) <= m.TARGET
model.C6_CibleFiscale = Constraint(rule=c6_rule)


### 4. SOLVE

In [ ]:
def resoudre(instance, solveur="cbc", verbose=True):
    opt = SolverFactory(solveur)
    return opt.solve(instance, tee=verbose)


In [ ]:
import shutil
chemin_cbc = shutil.which("cbc")
if chemin_cbc is None:
    import subprocess
    subprocess.run(["apt-get", "install", "-y", "coinor-cbc"], check=True)
    chemin_cbc = shutil.which("cbc")
assert chemin_cbc is not None, "CBC introuvable"
print(f"CBC disponible : {chemin_cbc}")


### 5. METTRE EN PLACE LA DATA

**15 références**, saisies directement dans le notebook. Deux profils cohabitent
délibérément :
- des références **en surstock** (le stock couvre largement plus que 5 mois de demande) :
  la stratégie optimale consistera à les laisser se consommer sans commander,
- des références **sous-couvertes** (le stock actuel ne suffit pas à couvrir la demande des
  5 prochains mois) : le modèle devra décider **quand** et **combien** commander, sous
  contrainte de MOQ et de fermeture fournisseur en août.


In [ ]:
noms_mois = {1:"Mai", 2:"Juin", 3:"Juillet", 4:"Août", 5:"Septembre"}
T_max  = 5
t_aout = [4]

# prix, LT (mois), MOQ, volume unitaire (m3), stock initial (mai), coût fixe de commande, profil
references = {
    "PN-01": dict(prix=150, LT=1, MOQ=100,  volume=0.050, stock_init=3000,  cout_commande=200, profil="Surstock"),
    "PN-02": dict(prix=60,  LT=1, MOQ=1200, volume=0.030, stock_init=2200,  cout_commande=150, profil="À réapprovisionner"),
    "PN-03": dict(prix=400, LT=1, MOQ=150,  volume=0.080, stock_init=2000,  cout_commande=400, profil="Surstock"),
    "PN-04": dict(prix=25,  LT=1, MOQ=1000, volume=0.010, stock_init=20000, cout_commande=100, profil="Surstock"),
    "PN-05": dict(prix=120, LT=2, MOQ=300,  volume=0.040, stock_init=7000,  cout_commande=250, profil="Surstock"),
    "PN-06": dict(prix=300, LT=2, MOQ=700,  volume=0.060, stock_init=650,   cout_commande=300, profil="À réapprovisionner"),
    "PN-07": dict(prix=90,  LT=1, MOQ=800,  volume=0.030, stock_init=1100,  cout_commande=180, profil="À réapprovisionner"),
    "PN-08": dict(prix=220, LT=2, MOQ=250,  volume=0.050, stock_init=2500,  cout_commande=280, profil="Surstock"),
    "PN-09": dict(prix=45,  LT=1, MOQ=1500, volume=0.020, stock_init=12000, cout_commande=120, profil="Surstock"),
    "PN-10": dict(prix=500, LT=3, MOQ=300,  volume=0.100, stock_init=260,   cout_commande=450, profil="À réapprovisionner"),
    "PN-11": dict(prix=70,  LT=1, MOQ=1000, volume=0.030, stock_init=6000,  cout_commande=160, profil="Surstock"),
    "PN-12": dict(prix=180, LT=2, MOQ=400,  volume=0.050, stock_init=610,   cout_commande=260, profil="À réapprovisionner"),
    "PN-13": dict(prix=35,  LT=1, MOQ=2000, volume=0.015, stock_init=15000, cout_commande=110, profil="Surstock"),
    "PN-14": dict(prix=260, LT=2, MOQ=400,  volume=0.060, stock_init=390,   cout_commande=320, profil="À réapprovisionner"),
    "PN-15": dict(prix=110, LT=1, MOQ=600,  volume=0.040, stock_init=4000,  cout_commande=190, profil="Surstock"),
}
refs = list(references.keys())

valeur_initiale = sum(references[i]["stock_init"]*references[i]["prix"] for i in refs)
print(f"Valeur de stock initiale (mai) : {valeur_initiale:,.0f} €")

TARGET = valeur_initiale - 2_000_000
print(f"Objectif de réduction         : 2 000 000 €")
print(f"Cible fiscale (fin septembre)  : {TARGET:,.0f} €")

df_ref = pd.DataFrame(references).T
df_ref.index.name = "PN"
display(df_ref)


**Demande mensuelle.** Les références « Surstock » ont une demande proportionnelle à leur
stock initial (~7,9%/mois, légère saisonnalité) : leur stock diminue mais reste largement
suffisant. Les références « À réapprovisionner » ont une demande fixée indépendamment de leur
stock initial, volontairement plus élevée que ce que leur stock de mai peut couvrir seul.


In [ ]:
variation = {1: 0.95, 2: 1.00, 3: 1.05, 4: 1.00, 5: 1.00}
taux_surstock = 0.079

demande_reappro = {"PN-02":380, "PN-06":115, "PN-07":190, "PN-10":47, "PN-12":105, "PN-14":68}

demande = {}
for i in refs:
    if references[i]["profil"] == "À réapprovisionner":
        base = demande_reappro[i]
    else:
        base = round(taux_surstock * references[i]["stock_init"])
    for t in range(1, T_max + 1):
        demande[(i, t)] = round(base * variation[t])

df_demande = pd.DataFrame({i: [demande[(i,t)] for t in range(1,T_max+1)] for i in refs},
                           index=[noms_mois[t] for t in range(1,T_max+1)]).T
print("Demande mensuelle par référence :")
display(df_demande)


**Stock de sécurité** (formule EOQ classique) :
$$SS_i = z_\alpha \cdot \sigma_{D,i} \cdot \sqrt{LT_i}$$


In [ ]:
z_alpha = 1.65   # niveau de service ~95%
sigma_D = {i: np.std([demande[(i,t)] for t in range(1, T_max+1)]) for i in refs}
SS = {i: round(z_alpha * sigma_D[i] * math.sqrt(references[i]["LT"])) for i in refs}

df_ss = pd.DataFrame({"Stock initial": {i: references[i]["stock_init"] for i in refs},
                       "SS": SS,
                       "LT (mois)": {i: references[i]["LT"] for i in refs},
                       "Profil": {i: references[i]["profil"] for i in refs}})
display(df_ss)


**Capacité de stockage** : fixée à 1,25× le volume nécessaire au stock initial.

In [ ]:
volume_initial = sum(references[i]["stock_init"]*references[i]["volume"] for i in refs)
V_max = round(volume_initial * 1.25)
print(f"Volume de stock initial : {volume_initial:,.0f} m³")
print(f"Capacité de stockage (V_max) : {V_max:,.0f} m³")

cout_stockage = {i: round(0.20 * references[i]["prix"] / 12, 4) for i in refs}   # tau=20%/an
M_bigM = max(references[i]["MOQ"] for i in refs) * 20


In [ ]:
data = DataPortal()
data['T'] = {None: T_max}
data['AoutSet'] = t_aout
data['I'] = refs
data['M'] = {None: M_bigM}

data['prix']          = {i: references[i]["prix"] for i in refs}
data['stock_init']    = {i: references[i]["stock_init"] for i in refs}
data['LT']            = {i: references[i]["LT"] for i in refs}
data['MOQ']           = {i: references[i]["MOQ"] for i in refs}
data['volume']        = {i: references[i]["volume"] for i in refs}
data['SS']            = SS
data['cout_stockage'] = cout_stockage
data['cout_commande'] = {i: references[i]["cout_commande"] for i in refs}
data['demande']       = demande
data['TARGET']        = {None: TARGET}
data['V_max']         = {None: V_max}

instance = model.create_instance(data)
print("Instance créée :", len(refs), "références,", T_max, "mois")


### Résolution

In [ ]:
resultats = resoudre(instance, solveur="cbc", verbose=True)
print("\nStatut :", resultats.solver.termination_condition)
print("Coût total optimal :", round(value(instance.cost), 2), "€")


---
## 6. Résultats et analyse


In [ ]:
valeur_finale = sum(references[i]["prix"]*value(instance.S[i, T_max]) for i in refs)
reduction = valeur_initiale - valeur_finale

print("=" * 55)
print("  BILAN GLOBAL")
print("=" * 55)
print(f"  Valeur de stock initiale (Mai)      : {valeur_initiale:,.0f} €")
print(f"  Valeur de stock finale (Septembre)  : {valeur_finale:,.0f} €")
print(f"  Réduction obtenue                   : {reduction:,.0f} € ({reduction/valeur_initiale*100:.1f}%)")
print(f"  Cible fiscale                       : {TARGET:,.0f} €")
print("  " + ("✅ Cible respectée" if valeur_finale <= TARGET else "❌ Cible dépassée"))
print("=" * 55)


### 6.1 Plan d'approvisionnement détaillé

In [ ]:
lignes = []
for i in refs:
    for t in range(1, T_max+1):
        lignes.append({
            "PN": i, "Profil": references[i]["profil"], "Mois": noms_mois[t],
            "Demande": demande[(i,t)],
            "Commande (O)": round(value(instance.O[i,t])),
            "Stock fin de mois (S)": round(value(instance.S[i,t])),
            "z (commande passée)": int(round(value(instance.z[i,t]))),
        })
df_resultats = pd.DataFrame(lignes)

df_commandes = df_resultats[df_resultats["z (commande passée)"] == 1][
    ["PN", "Profil", "Mois", "Commande (O)"]
].reset_index(drop=True)

print(f"Nombre de commandes passées sur l'horizon : {len(df_commandes)}")
display(df_commandes)


### 6.2 Détail par référence

In [ ]:
for i in refs:
    print(f"\n--- {i} ({references[i]['profil']}) ---")
    display(df_resultats[df_resultats["PN"] == i].drop(columns=["PN","Profil"]).set_index("Mois"))


### 6.3 Graphique — trajectoire globale de la valeur de stock

In [ ]:
import matplotlib.pyplot as plt

valeurs_mois = [sum(references[i]["prix"]*value(instance.S[i,t]) for i in refs) for t in range(1, T_max+1)]

fig, ax = plt.subplots(figsize=(9,5))
ax.plot([noms_mois[t] for t in range(1,T_max+1)], valeurs_mois, marker="o", linewidth=2.5,
        color="steelblue", label="Valeur de stock (optimisée)")
ax.axhline(TARGET, color="red", linestyle="--", label=f"Cible ({TARGET:,.0f} €)")
ax.axhline(valeur_initiale, color="gray", linestyle=":", label=f"Valeur initiale ({valeur_initiale:,.0f} €)")
ax.set_title("Trajectoire de la valeur de stock (Mai → Septembre)")
ax.set_xlabel("Mois"); ax.set_ylabel("Valeur du stock (€)")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


### 6.4 Graphique — valeur de stock par profil (Surstock vs À réapprovisionner)

In [ ]:
profils = ["Surstock", "À réapprovisionner"]
valeurs_par_profil = {p: [] for p in profils}
for t in range(1, T_max+1):
    for p in profils:
        v = sum(references[i]["prix"]*value(instance.S[i,t]) for i in refs if references[i]["profil"]==p)
        valeurs_par_profil[p].append(v)

fig, ax = plt.subplots(figsize=(9,5))
labels = [noms_mois[t] for t in range(1,T_max+1)]
ax.bar(labels, valeurs_par_profil["Surstock"], label="Surstock")
ax.bar(labels, valeurs_par_profil["À réapprovisionner"],
       bottom=valeurs_par_profil["Surstock"], label="À réapprovisionner")
ax.set_title("Valeur de stock par profil de référence")
ax.set_xlabel("Mois"); ax.set_ylabel("Valeur du stock (€)")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()


### 6.5 Graphique — commandes passées dans le temps (respect de la fermeture d'août)

In [ ]:
fig, ax = plt.subplots(figsize=(9,4.5))
pivot_cmd = df_commandes.pivot_table(index="PN", columns="Mois", values="Commande (O)", aggfunc="sum")
pivot_cmd = pivot_cmd.reindex(columns=[noms_mois[t] for t in range(1,T_max+1)])
pivot_cmd.plot(kind="barh", stacked=True, ax=ax, colormap="viridis")
ax.set_title("Commandes passées par référence et par mois")
ax.set_xlabel("Quantité commandée"); ax.set_ylabel("PN")
ax.axvline(0, color="black", linewidth=0.8)
ax.legend(title="Mois", bbox_to_anchor=(1.02,1), loc="upper left")
plt.tight_layout(); plt.show()

print("Aucune commande n'apparaît en Août : la contrainte C4 (fermeture fournisseur) est respectée.")


### 6.6 Graphique — utilisation de la capacité de stockage

In [ ]:
volumes_mois = [sum(references[i]["volume"]*value(instance.S[i,t]) for i in refs) for t in range(1, T_max+1)]

fig, ax = plt.subplots(figsize=(9,5))
ax.bar([noms_mois[t] for t in range(1,T_max+1)], volumes_mois, color="seagreen", label="Volume utilisé")
ax.axhline(V_max, color="red", linestyle="--", label=f"Capacité V_max ({V_max:,.0f} m³)")
ax.set_title("Utilisation de la capacité de stockage")
ax.set_xlabel("Mois"); ax.set_ylabel("Volume (m³)")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

taux_occupation = [v/V_max*100 for v in volumes_mois]
print("Taux d'occupation de l'entrepôt par mois :",
      {noms_mois[t]: f"{taux_occupation[t-1]:.1f}%" for t in range(1,T_max+1)})


### 6.7 Analyse et enseignements

- **Le désengagement porte l'essentiel de la réduction.** Les 9 références « Surstock »
  contribuent à elles seules à la quasi-totalité de la baisse de valeur : en ne passant
  **aucune commande**, leur stock se consomme naturellement sur les 5 mois, ce qui est optimal
  car le coût de possession croît avec le niveau de stock détenu.

- **Le réapprovisionnement reste nécessaire pour certaines références.** Les 6 références
  « À réapprovisionner » (PN-02, PN-06, PN-07, PN-10, PN-12, PN-14) ont un stock initial
  insuffisant pour couvrir la demande des 5 prochains mois : le modèle décide, pour chacune,
  **une seule commande**, dimensionnée au minimum requis par le MOQ, et placée **le plus tard
  possible** pour minimiser le coût de possession.

- **La fermeture fournisseur en août influence réellement le calendrier.** Pour au moins une
  référence (PN-02), la date optimale sans contrainte de fermeture aurait été août ; la
  contrainte C4 oblige à anticiper la commande en juillet — un exemple concret de l'impact
  opérationnel de cette contrainte sur la décision d'achat.

- **La capacité de stockage n'est jamais le facteur limitant** sur ce scénario : le taux
  d'occupation maximal reste sous la barre des 100% (voir 6.6), la marge provient
  essentiellement de la baisse continue du stock au fil des mois. La contrainte C5 reste
  néanmoins active dans le modèle et deviendrait contraignante avec un stock initial plus élevé
  ou une capacité d'entrepôt réduite.

- **La cible de réduction (~2 M€) est atteinte** tout en respectant simultanément le stock de
  sécurité, le MOQ, la fermeture fournisseur et la capacité de stockage — démontrant qu'un
  objectif de désengagement de trésorerie immobilisée en stock peut être conciliable avec la
  continuité de service, à condition d'arbitrer finement entre les références à liquider et
  celles à réapprovisionner.
